In [1]:
import pickle
import numpy as np
import os
import tqdm
import torch
import gc

from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()

from importlib import reload
from matplotlib import pyplot as plt

import matrix_approx_zeshel

matrix_approx_zeshel = reload(matrix_approx_zeshel)

!ls  | grep matrix_approx_zeshel

2023-11-15 00:52:14.889841: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-11-15 00:52:15.090258: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-11-15 00:52:15.094079: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-11-15 00:52:16.958094: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


matrix_approx_zeshel.py


# Open Data loader & context

In [2]:
def load_ment_to_ent_scores(directory = "yugioh", shuffle_rows = 0, full = True):
    data = list()

    for file in os.listdir(directory):
        path = f"{directory}/{file}"
        print(f"Loading file {path}")
        with open(path, "rb") as f:
            data.append(
                pickle.load(f)
            )
    data = sorted(data, key = lambda x: x["arg_dict"]["n_ment_start"])

    for i in range(len(data) - 1):
        if full:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] == data[i + 1]["arg_dict"]["n_ment_start"]
        else:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] <= data[i + 1]["arg_dict"]["n_ment_start"]
        
    ment_to_ent_scores = list(map(lambda x: x["ment_to_ent_scores"], data))
    ment_to_ent_scores = np.vstack(ment_to_ent_scores)
    print("Loaded shape = ", ment_to_ent_scores.shape)
    
    if shuffle_rows:
        print(f"Shuffling... (seed = {shuffle_rows})")
        np.random.seed(shuffle_rows)
        np.random.shuffle(ment_to_ent_scores)
    
    return ment_to_ent_scores

In [3]:
from sklearn.cluster import KMeans

def from_labels(X, labels):
    K = list()
    for i in range(100):
        sl_i = np.arange(X.shape[0])[labels == i]
        sl = X[labels == i]
        
        if len(sl_i) == 0:
            # bug-mode
            while True:
                chosen = np.random.choice(np.arange(X.shape[0])[labels < i], size=1)[0]
                if chosen not in K:
                    K.append(chosen)
                    break
            continue
            
        center = sl.mean(axis=0).reshape(1, -1)
        best = euclidean_distances(sl, center).argmin()
        K.append(sl_i[best])
        assert labels[K[-1]] == i
    return K

class EvalContextRelevs:
    def __init__(self, relevs, key_size = 100, train_size = 0.7, key_games = None, seed = 17, det_attempts = 0):
        self.relevs = np.array(relevs)
        self.reqs_count = self.relevs.shape[0]
        self.games_count = self.relevs.shape[1]
        
        self.key_size = key_size
        self.key_games = (
            np.random.choice(np.arange(self.games_count), key_size, replace=False)
            if key_games is None else
            key_games
        )
        np.random.seed(seed)
        #np.random.shuffle(self.requests)

        self.try_det_attempts(det_attempts)
        self.train_split = int(self.reqs_count * train_size)

        assert key_size + 1 < self.train_split


        self.key_relevs = self.relevs[:key_size]
        self.train_relevs = self.relevs[key_size + 1: self.train_split]
        self.test_relevs = self.relevs[self.train_split:]

        self.slices = ["key", "train", "test"]
        print(len(self.key_relevs), len(self.train_relevs), len(self.test_relevs))

    def get_top_games(self):
        return self.relevs.mean(axis=0).argsort()[:100]

    def set_top_games_as_key(self):
        self.key_games = self.get_top_games()
        return self

    def get_kmeans_games(self, all_from_labels=True):
        X = self.get_relevs("train").T

        k_func = (
            (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
            if not all_from_labels else
            (lambda C : from_labels(X, C.labels_))
        )
        K_KMeans = k_func(
            KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
        )
        
        return K_KMeans
    
    def set_kmeans_games_as_key(self, *args, **kwargs):
        self.key_games = self.get_kmeans_games(*args, **kwargs)
        return self
    

    def try_det_attempts(self, det_attempts):
        def get_det(r):
            kr = r[:self.key_size, self.key_games] - r.mean()
            return np.abs(np.linalg.det(kr))

        best_i_array = np.arange(len(self.relevs))

        for i in range(det_attempts):

            r_i_array = np.arange(len(self.relevs))
            np.random.shuffle(r_i_array)

            n, o = get_det(self.relevs[r_i_array, :]), get_det(self.relevs[best_i_array, :])
            
            # print(f"try update key_reqs ({o} vs {n}...")
            if n > o:
                best_i_array = r_i_array
                print(f"updated det ({i}, {o} -> {n})")

        print("Best det = ", get_det(self.relevs[best_i_array, :]))

        self.relevs = self.relevs[best_i_array, :]
        print("Current de = ", get_det(self.relevs))
        
    def get_relevs(self, t = "train"):
        if t == "train":
            return self.train_relevs
        elif t == "key":
            return self.key_relevs
        elif t == "test":
            return self.test_relevs
        else:
            assert False
            
    def get_requests(self, t = "train"):
        if t == "train":
            return self.train_reqs
        elif t == "key":
            return self.key_reqs
        elif t == "test":
            return self.test_reqs
        else:
            assert False

# Games Data loader & context

In [4]:
import collections
import pickle
import numpy as np
import tqdm
import os
import gc

import matplotlib.pyplot as plt


def load(limit, raw_path = "stand/log.local.logtime2.txt", path = "log.local.logtime2.bin", key_games = None, seed = 17, det_attempts = 0):
    readvector = lambda s : np.array(list(map(float, s.strip()[1:-2].split(","))))
    requests = list()
    docembs = collections.defaultdict(dict)

    if os.path.isfile(path):
        with open(path, "rb") as f:
            flimit, frequests, fdocembs = pickle.load(f)
            if flimit == limit:
                requests, docembs = frequests, fdocembs
            else:
                print(f"WARN: buffered limit is different, {flimit} != {limit}, reloading...")

    if not requests:
        with open(raw_path) as f:
            req = list()
            reqid = None
            models = list()
            prevreqmodel = None
            reqmodel = dict()
            prevmodelid = -1
            bannermodelid = -1
            for i, line in tqdm.tqdm_notebook(enumerate(f)):
                if line.startswith("Model = 6;"):
                    prevreqmodel = reqmodel
                    reqmodel = dict()

                if line.startswith("Model = "):
                    spl = line.split(" ")
                    prevmodelid = int(spl[2][:-1])
                    bannermodelid = max(bannermodelid , prevmodelid)
                    reqmodel[prevmodelid] = readvector(spl[3])
                elif line.startswith("dbid"):
                    spl = line.split(" ")
                    dbid = int(spl[1][:-1])
                    docembs[bannermodelid][dbid] = readvector(spl[2])
                elif line.startswith("seed"):
                    if len(requests) >= limit:
                        break
                    if req:
                        requests.append((reqid, prevreqmodel, sorted(req)))
                        req = list()
                    reqid = "$_" + (line.split()[1] + "_" + line.split()[3])
                else:
                    req.append(
                        (int(line.split()[0]), float(line.split()[1]))
                    )
        
        with open(path, "wb") as f:
            pickle.dump((limit, requests, docembs), f)

    games_count = len(requests[0][2])
    assert games_count == 16514
    requests = [r for r in requests if len(r[2]) == games_count]
    
    print([(i, len(docembs[i].keys())) for i in docembs])  # should be equal
    docblocks = {
        mid : np.array([x[1] for x in sorted(list(docembs[mid].items()))])
        for mid in docembs
    }
    
    class EvalContext:
        def __init__(self, games_count = games_count, key_size = 100, train_size = 0.7, key_games = None, seed = 17, det_attempts = 0):
            self.games_count = games_count
            
            self.key_size = key_size
            self.key_games = (
                np.random.choice(np.arange(games_count), key_size, replace=False)
                if key_games is None else
                key_games
            )

            self.requests = requests
            np.random.seed(seed)
            np.random.shuffle(self.requests)
            
            self.try_det_attempts(det_attempts)

            self.key_reqs = self.requests[:key_size + 1]
            self.key_reqs_idx = np.arange(key_size + 1)

            self.train_split = int(len(self.requests) * train_size)

            assert key_size + 1 < self.train_split

            self.train_reqs = self.requests[key_size + 1: self.train_split]
            self.test_reqs = self.requests[self.train_split:]

            self.slices = ["key", "train", "test"]
            print(len(self.key_reqs), len(self.train_reqs), len(self.test_reqs))

            self.docblocks = docblocks
            self.relevs = dict()
            
        def get_top_games(self):
            if not hasattr(self, "top_games"):
                embed_games = np.array([
                    np.array([r[2][g_i][1] for r in self.get_requests("train")])
                    for g_i in range(self.games_count)
                ])

                self.embed_games_mean = embed_games.mean(axis=1)
                self.top_games_all = (-self.embed_games_mean).argsort()
                self.top_games = self.top_games_all[:len(self.key_games)]

            return self.top_games
        
        def set_top_games_as_key(self):
            self.key_games = self.get_top_games()
            return self
        
        def get_kmeans_games(self, all_from_labels=True):
            X = self.get_relevs("train").T

            k_func = (
                (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
                if not all_from_labels else
                (lambda C : from_labels(X, C.labels_))
            )
            K_KMeans = k_func(
                KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
            )

            return K_KMeans

        def set_kmeans_games_as_key(self, *args, **kwargs):
            self.key_games = self.get_kmeans_games(*args, **kwargs)
            return self

        def try_det_attempts(self, det_attempts, model_id = 6):
            def get_det(r, r_i_array):
                kr = np.array([
                    r[r_i][1][model_id]
                    for r_i in r_i_array[:100]
                ])
                return np.abs(np.linalg.det(kr[:kr.shape[1], :]))

            best_i_array = np.arange(len(self.requests))

            for _ in range(det_attempts):
                # print("try update key_reqs...")
                
                r_i_array = np.arange(len(self.requests))
                np.random.shuffle(r_i_array)
                
                n, o = get_det(self.requests, r_i_array), get_det(self.requests, best_i_array)
                # print(n, o)
                if n > o:
                    best_i_array = r_i_array
                    # print("updated!")

            print("Best det = ", get_det(self.requests, best_i_array))
            
            new_requests = [
                self.requests[i]
                for i in best_i_array
            ]
            
            del self.requests
            gc.collect()

            self.requests = new_requests
            print(get_det(self.requests, np.arange(len(self.requests))))

        def get_relevs(self, t = "train"):
            if t not in self.relevs:
                self.relevs[t] = np.array([
                    np.array([g_i[1] for g_i in r[2]])
                    for r in self.get_requests(t)
                ])
                
            return self.relevs[t]

        def get_requests(self, t = "train"):
            if t == "train":
                return self.train_reqs
            elif t == "key":
                return self.key_reqs
            elif t == "test":
                return self.test_reqs
            else:
                assert False

    return EvalContext(key_games = key_games, seed = seed, det_attempts = det_attempts)

# Models

In [5]:
class Popular:
    def __init__(self, ctx):
        self.ctx = ctx
        self.game_avg_scores = {
            t : self.get_user_scores(t).mean(axis = 0)
            for t in self.ctx.slices
        }
        self.trus_top = dict()
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        return [self.top_logits for _ in self.get_user_scores(t)]
    
    def get_score(self, t, topsize = 100):    
        recs = self.recommend(t)
        
        if isinstance(recs, list):
            recs = np.array(recs)
        
        mse = np.mean((recs - self.get_user_scores(t)) ** 2)
        
        if isinstance(recs, tf.Tensor):
            recs = tf.argsort(-recs, axis=1)[:, :topsize].numpy()
        else:
            recs = np.argsort(-recs, axis=1)[:, :topsize]
        
        if (t, topsize) not in self.trus_top:
            trus = self.get_user_scores(t)
            trus = np.argsort(-trus, axis=1)[:, :topsize]
            self.trus_top[(t, topsize)] = trus
            
        trus = self.trus_top[(t, topsize)]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

In [6]:
import tensorflow as tf
import copy
import tensorflow_ranking as tfr

DEFAULT_FIT_KWARGS = {
    "learning_rate": 0.001,
    "n": 500,
    "c": 5000,
    "matrix_l2": 0,
    "dssm_l2": 0,
    "train_popbias": False,
    "train_bias": False,
    "train_vbias": False,
    "verbose": True,
    "train_matrix": True,
    "train_dssm": False,
    "loss": "mse",
    "ubatch": 1e9,
    "activation": "relu",
    "score_verbose": 0,
    "trainable_items": False,
    "use_keys_in_train": False,
    "Winit": "glorot0.01",
    "Wfreeze": False,
    "TEinit": "legacy",
    "normalize_embs": True
}

class CBKnnV0(Popular):
    def __init__(self, ctx, fit_kwargs = dict()):
        super().__init__(ctx)
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(fit_kwargs)
        self.fit_kwargs = p
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        
        self.embed_users = {
            t : np.array([
                    np.array([r_i[i] for i in sorted(ctx.key_games)])
                    for r_i in ctx.get_relevs(t)
                ])
            for t in ctx.slices
        }
        self.embed_users_mean = {
            t : self.embed_users[t].mean(axis = 0)
            for t in ctx.slices
        }
        # embed_users = embed_users - embed_users_mean
        print("self.embed_users['train'].shape = ", self.embed_users['train'].shape )
        
        self.embed_games = np.array([
            np.array([r[g_i] for r in ctx.get_relevs("key")])
            for g_i in range(ctx.games_count)
        ])
        
        self.embed_games_mean = self.embed_games.mean(axis = 0)
        
        # embed_games = embed_games - embed_games_mean
        print("self.embed_games.shape = ", self.embed_games.shape)
        
        
        if self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["TEinit"] == "legacy":
                value = self.embed_games
            elif self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                value = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print(type(value))
            else:
                assert False, "unknown TEinit: " + str(self.fit_kwargs["TEinit"])
                
            self.trainable_games = tf.Variable(
                tf.convert_to_tensor(value, dtype=tf.float32),
                trainable=True
            )
        else:
            if self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                self.embed_games = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print("ANNCur init")
            
        
        self.games_top_key = self.embed_games.mean(axis = 1)
        
        self.games2users = np.array([
            self.embed_games[g_i]
            for g_i in ctx.key_games
        ])
        print("self.games2users.shape = ", self.games2users.shape)
        
        self.core_users_scores = ctx.get_relevs("key")
        print("self.core_users_scores.shape = ", self.core_users_scores.shape)
        
        self.core_users_embs = self.embed_users["key"]
        print("self.core_users_embs.shape = ", self.core_users_embs.shape)
        
        self.ge_users = (
            (self.embed_users["train"].T / self.embed_users["train"].mean(axis=1)).T @ self.games2users
        )
        # ge_users = embed_users @ games2users
        print("self.ge_users.shape = ", self.ge_users.shape)
    
    def __repr__(self):
        return "CbKnn(" + str(self.fit_kwargs) + ")"
        
    # inherited   
    # def get_user_scores(self, t):
    
    def get_user_embs(self, t):
        if self.fit_kwargs["normalize_embs"]:
            return (self.embed_users[t] - self.embed_users_mean["train"]) / self.embed_users[t].std(axis=0)
        else:
            return self.embed_users[t]
    
    def get_game_embs(self):
        if not self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["normalize_embs"]:
                return (self.embed_games - self.embed_games.mean(axis=0)) / self.embed_games.std(axis=0)
            else:
                return self.embed_games 
        else:
            return self.trainable_games

    def fit(self, **kwargs):
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        p.update(kwargs)

        self.p = p
        self.train_bias = p["train_bias"]
        self.train_vbias = p["train_vbias"]
        self.train_popbias = p["train_popbias"]
        self.train_matrix = p["train_matrix"]
        self.train_dssm = p["train_dssm"]
        self.loss = p["loss"]

        if p["use_keys_in_train"]:
            train_user_scores = np.vstack([
                self.get_user_scores("key"),
                self.get_user_scores("train")
            ])
            train_user_embs = np.vstack([
                self.get_user_embs("key"),
                self.get_user_embs("train")
            ])
        else:
            train_user_scores = self.get_user_scores("train")
            train_user_embs = self.get_user_embs("train")

        game_embs = self.get_game_embs()
        
        
        
        if self.p["Winit"] == "glorot0.01":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
            values = values / 100.  
        elif self.p["Winit"] == "glorot":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
        elif self.p["Winit"] == "eye":
            values = np.eye(train_user_embs.shape[1], game_embs.shape[1])
        else:
            assert False, "init is not known:" + str(self.p["init"])
            
        self.W = tf.Variable(values, trainable = True) 
        self.pb = tf.Variable(1., trainable = True) 
        self.b = tf.Variable(0., trainable = True) 
        
        if p["verbose"]:
            print("self.W = ", self.W)
            print("0-loss = ", tf.reduce_mean((train_user_scores - 0) ** 2))
    
        opt =  tf.keras.optimizers.Adam(learning_rate=p["learning_rate"])

        if self.train_dssm:
            self.train_dssm = True

            dim = game_embs.shape[1]
            self.g_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.g_dssm(game_embs)
            
            # dim = train_user_embs.shape[1]
            self.u_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.u_dssm(train_user_embs)
            
        if self.train_vbias:
            self.vb = tf.Variable(
                np.zeros_like(self.game_avg_scores["train"], dtype=np.float32),
                trainable = True
            ) 
        
        
        for i in range(p["n"]):
            def loss():
                def get_logits_scores(loss_batch = 1e9):
                    game_slice = None
                    
                    ubatch = p["ubatch"]
                    if ubatch >= train_user_scores.shape[0]:
                        train_user_embs_ = train_user_embs
                        scores_ = train_user_scores
                    else:
                        u_slice = np.random.choice(np.arange(train_user_scores.shape[0]), ubatch, replace = True)
                        train_user_embs_ = train_user_embs[u_slice]
                        scores_ = train_user_scores[u_slice]
                    
                    if loss_batch >= game_embs.shape[0]:
                        game_embs_ = game_embs
                        scores_ = scores_
                    else:
                        game_slice = np.random.choice(np.arange(game_embs.shape[0]), loss_batch, replace = True)
                        game_embs_ = (
                            game_embs[game_slice]
                            if isinstance(game_embs, np.ndarray) else
                            tf.gather(game_embs, game_slice)
                        )
                        scores_ = scores_[:, game_slice]
                        
                    
                    logits = 0
                    
                    if self.train_matrix:
                        logits += train_user_embs_ @ self.W @ tf.transpose(game_embs_)

                    if self.train_dssm:
                        logits += self.u_dssm(train_user_embs_) @ tf.transpose(self.g_dssm(game_embs_))
                    
                    if self.train_popbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.pb * self.game_avg_scores["train"]
                        else:
                            logits += self.pb * self.game_avg_scores["train"][game_slice]
                        
                    if self.train_vbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.vb
                        else:
                            logits += tf.gather(self.vb, game_slice)
                        
                    if self.train_bias:
                        logits += self.b
                        
                    return logits, scores_
                        
                if self.loss == "mse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    v = tf.reduce_mean((scores - logits) ** 2)
                elif self.loss == "qmse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    q_mean = scores.mean(axis=1)
                    v = tf.reduce_mean(((scores.T - q_mean).T - logits) ** 2)
                elif self.loss == "ApproxNDCGLoss":
                    while True:
                        logits, scores = get_logits_scores(
                            loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))

                        v = tfr.keras.losses.ApproxNDCGLoss()(
                            scores.astype(np.float32),
                            logits
                        )
                    
                        if not tf.math.is_nan(v).numpy().any():
                            break
                        else:
                            if p["verbose"]:
                                print("nanloss ignored")
                elif self.loss == "softmax":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ))
                elif self.loss == "softmax_signed":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    sf = tf.nn.softmax(logits, axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        sf,
                        -sf
                    ))
                elif self.loss == "softmax_weighted":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ) * scores)
                elif self.loss == "sigmoid_top_100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    mask = np.argsort(-scores, axis=1) < 100
                    v = -tf.reduce_sum(
                        tf.math.log_sigmoid((2 * mask - 1) * logits)
                    )
                elif self.loss == "KL100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    mask = np.argsort(-scores, axis=1) < 100
                    v = tf.keras.losses.KLDivergence()((scores.T >= qs.T).T, logits)
                else:
                    assert False
                    
                if self.p["c"]:
                    v += tf.reduce_mean(self.W * self.W) * p["c"]
                    
                if self.p["dssm_l2"]:
                    for weights_ in [self.u_dssm.weights, self.g_dssm.weights]:
                        for weight_ in weights_:
                            v += tf.reduce_sum(weight_ * weight_) * self.p["dssm_l2"]
                
                if self.p["matrix_l2"]:
                    v += tf.reduce_sum(self.W * self.W) * self.p["matrix_l2"]
                
                if p["verbose"]:
                    print(v.numpy())
                
                return v

            weights = list()
            
            if self.train_matrix and (not self.p["Wfreeze"]):
                weights += [self.W]

            if self.train_dssm:
                weights += self.u_dssm.weights
                weights += self.g_dssm.weights
            
            if self.train_popbias:
                weights += [self.pb]

            if self.train_vbias:
                weights += [self.vb]   
                
            if self.train_bias:
                weights += [self.b]   
                
            
            if self.p["trainable_items"]:
                weights += [self.trainable_games]
                
                
            opt.minimize(loss, weights)
            
            if p["score_verbose"] and (i % p["score_verbose"] == 0):
                print(f"\n=== Iteration {i} ===")
                for sl in self.ctx.slices:
                    print(f"slice = {sl}, score = {self.get_score(sl)}")
        print("last loss = ", loss())

    def recommend(self, t):
        logits = 0
                    
        if self.train_matrix:
            logits += self.get_user_embs(t) @ self.W @ tf.transpose(self.get_game_embs())

        if self.train_dssm:
            logits += self.u_dssm(self.get_user_embs(t)) @ tf.transpose(self.g_dssm(self.get_game_embs()))

        if self.train_popbias:
            logits += self.pb * self.game_avg_scores["train"]       
            
        if self.train_vbias:
            logits += self.vb
            
        if self.train_bias:
            logits += self.b
            
        return logits
    
    # inherited
    # def get_score(self, t, topsize = 100):
    
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))

In [7]:
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))

In [8]:
class AnnCUR(Popular):
    def __init__(self, ctx, oracle=False, key_games=None, name=None):
        super().__init__(ctx)
        self.oracle = oracle
        self.ctx = ctx
        self.name = name

        self.key_cols_idx = np.array(sorted(ctx.key_games if key_games is None else key_games))
        rows_idx = np.arange(ctx.get_relevs("train").shape[0])
        
        self.cur = matrix_approx_zeshel.CURApprox(
            rows=torch.from_numpy(ctx.get_relevs("train")),
            cols=torch.from_numpy(ctx.get_relevs("train")[:, self.key_cols_idx]),
            row_idxs=rows_idx,
            col_idxs=self.key_cols_idx,
            approx_preference="rows",
            A=(torch.from_numpy(ctx.get_relevs("train")) if oracle else None)
        )        
        
    def __repr__(self):
        if self.name is None:
            return f"AnnCur({id(self)})"
        else:
            return f"AnnCur({self.name} - {id(self)})"
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        key_relevs = torch.from_numpy(self.ctx.get_relevs(t)[:, self.key_cols_idx])
        return self.cur.get_complete_row(key_relevs)
    
    def get_score(self, t, topsize = 100):       
        recs = np.array(self.recommend(t))
        trus = self.get_user_scores(t)

        mse = np.mean((recs - trus) ** 2)

        recs = np.argsort(-recs, axis=1)[:, :topsize]
        trus = np.argsort(-trus, axis=1)[:, :topsize]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

# Evals

In [9]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances

def from_labels(X, labels):
    K = list()
    for i in range(100):
        sl_i = np.arange(X.shape[0])[labels == i]
        sl = X[labels == i]
        
        if len(sl_i) == 0:
            # bug-mode
            while True:
                chosen = np.random.choice(np.arange(X.shape[0])[labels < i], size=1)[0]
                if chosen not in K:
                    K.append(chosen)
                    break
            continue
            
        center = sl.mean(axis=0).reshape(1, -1)
        best = euclidean_distances(sl, center).argmin()
        K.append(sl_i[best])
        assert labels[K[-1]] == i
    return K

def K_by_min(X):
    center = X.mean(axis=0)
    K = [euclidean_distances(X, center.reshape(1, -1)).argmax()]

    while len(K) < 100:
        K.append(euclidean_distances(X, X[K]).min(axis=1).argmax())
    return K


def eval_clustering(ctx, all_from_labels=False):
    X = ctx.get_relevs("train").T
    from sklearn.cluster import KMeans
    
    k_func = (
        (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
        if not all_from_labels else
        (lambda C : from_labels(X, C.labels_))
    )
    
    K_KMeans = k_func(
        KMeans(n_clusters=100, random_state=0).fit(X)  #, n_init="auto").fit(X)
    )


    from sklearn.cluster import BisectingKMeans
    K_BisectingKMeans = k_func(
        BisectingKMeans(n_clusters=100, random_state=0).fit(X)
    )


    from sklearn.cluster import MiniBatchKMeans
    K_MiniBatchKMeans = from_labels(
        X,
        MiniBatchKMeans(n_clusters=100, random_state=0).fit(X).labels_
    )

    from sklearn.cluster import AgglomerativeClustering
    K_AgglomerativeClustering = from_labels(
        X,
        AgglomerativeClustering(n_clusters=100).fit(X).labels_
    )

    from sklearn.cluster import SpectralCoclustering
    K_SpectralCoclustering = from_labels(
        X,
        SpectralCoclustering(n_clusters=100, random_state=0).fit(X).row_labels_
    )

    from sklearn.cluster import SpectralBiclustering
    K_SpectralBiclustering = from_labels(
        X,
        SpectralBiclustering(n_clusters=100, random_state=0).fit(X).row_labels_
    )

    from sklearn.cluster import SpectralClustering
    K_SpectralClusteringNN = from_labels(
        X,
        SpectralClustering(n_clusters=100, random_state=0, affinity="nearest_neighbors", n_jobs=-1).fit(X).labels_
    )

    K_ByMin = K_by_min(X)

    ev([
        Popular(ctx),
        AnnCUR(ctx),
        AnnCUR(ctx, key_games=np.arange(100), name="key_games=np.arange(100)"),
        AnnCUR(ctx, key_games=np.random.choice(np.arange(X.shape[0]), size=100, replace=False), name="random"),
        AnnCUR(ctx, key_games=K_KMeans, name="K_KMeans"),
        AnnCUR(ctx, key_games=K_BisectingKMeans, name="K_BisectingKMeans"),
        AnnCUR(ctx, key_games=K_MiniBatchKMeans, name="K_MiniBatchKMeans"),
        AnnCUR(ctx, key_games=K_AgglomerativeClustering, name="K_AgglomerativeClustering"),
        AnnCUR(ctx, key_games=K_SpectralCoclustering, name="K_SpectralCoclustering"),
        AnnCUR(ctx, key_games=K_SpectralBiclustering, name="K_SpectralBiclustering"),
        AnnCUR(ctx, key_games=K_SpectralClusteringNN, name="K_SpectralClusteringNN"),
        AnnCUR(ctx, key_games=K_ByMin, name="K_ByMin"),
    ])

In [10]:
DATASETS = ["yugioh", "pro_wrestling", "star_trek", "doctor_who", "military"]

# support items choice

In [11]:
# setup:
# 1000 items, 1000 + 1000 queries
recall_k = 5
n_runs = 10
n_items_per_run = 1000
test_item_cnt = 100
item_cnt_range = list(range(5, 101, 5))
assert test_item_cnt in item_cnt_range

In [12]:
from collections import defaultdict
from itertools import combinations

from matplotlib import pyplot as plt
import numpy as np
import os
from sklearn.metrics import top_k_accuracy_score
from sklearn.cluster import KMeans
from scipy.stats import mannwhitneyu, ttest_rel

In [13]:
#with open("collections_10000_items/test.npy", "rb") as fin:
#    all_test = np.load(fin)
#with open("collections_10000_items/train.npy", "rb") as fin:
#    all_train = np.load(fin)

#assert all_train.shape[0] == n_runs * n_items_per_run
#assert all_test.shape[0] == n_runs * n_items_per_run


In [14]:
def eval_items(items, train, test):
    item_embeds = train.dot(np.linalg.pinv(train[items]))
    approx_test_rel = item_embeds.dot(test[items])
    approx_train_rel = item_embeds.dot(train[items])
    best_items = np.argsort(test, axis=0)[-1]
    recall = top_k_accuracy_score(best_items, approx_test_rel.T, k=recall_k, labels=np.arange(train.shape[0]))
    test_mse = np.mean((test - approx_test_rel) ** 2)
    train_mse = np.mean((train - approx_train_rel) ** 2)
    return recall, test_mse, train_mse

def get_rnd_items(n_items, train):
    return np.random.choice(train.shape[0], n_items, replace=False)

def get_kmeans_items(n_items, train):
    kmeans = KMeans(n_clusters=n_items, n_init="auto").fit(train)
    items = []
    for center in kmeans.cluster_centers_:
        distances = ((train - center) ** 2).sum(axis=1)
        assert distances.shape == (train.shape[0],)
        for item_id in np.argsort(distances):
            if item_id not in items:
                items.append(item_id)
                break
    return np.array(items, dtype=np.int64)

greedy_items = []
def get_greedy(n_items, train, lazy=True):
    global greedy_items
    if not lazy:
        greedy_items = []
    if len(greedy_items) >= n_items:
        return np.array(greedy_items[:n_items], dtype=np.int64)
    
    gram = train.dot(train.T)
    gram /= gram.mean()
    items = []
    remaining_items = np.ones(train.shape[0], dtype="bool")
    for t in range(n_items):
        scores = (gram ** 2).sum(axis=1)
        assert np.allclose(scores[~remaining_items], np.zeros_like(scores[~remaining_items])), (
            t, scores[~remaining_items]
        )
        scores[remaining_items] /= gram[remaining_items, remaining_items]
        if max(scores) < 1e-9:
            break
        new_item = scores.argmax()
        assert remaining_items[new_item]
        coefs = gram[new_item] / np.sqrt(gram[new_item, new_item])
        diff = coefs.reshape(-1, 1).dot(coefs.reshape(1, -1))
        gram -= diff
        assert np.allclose(gram[new_item], np.zeros_like(gram[new_item]), atol=1e-6), (
            t, gram[new_item].std()
        )
        items.append(new_item)
        remaining_items[new_item] = False
    assert len(items) == n_items
    greedy_items.clear()
    greedy_items.extend(items)
    return np.array(items, dtype=np.int64)


### fast-a

In [15]:
def coitem_algorithm(n_support, candidate_items, target_items, min_item_rel_norm=1e-5, eps=1e-6):
    support_items = []
    support_items_scores = []
    n_c, n_q = candidate_items.shape
    n_t = target_items.shape[0]

    candidate_item_squared_norms = (candidate_items ** 2).sum(axis=1)
    min_item_norm = min_item_rel_norm * np.sqrt(candidate_item_squared_norms.mean())
    
    candidate_coitems = candidate_items.dot(
        target_items.T.dot(target_items)
    )
    orthonormed_support_items = np.zeros((n_support, n_q))
    remaining_items = np.ones(candidate_items.shape[0], dtype="bool")
    for t in tqdm.tqdm_notebook(range(n_support)):
        scores = (candidate_coitems * candidate_items).sum(axis=1)
        remaining_items &= (candidate_item_squared_norms >= min_item_norm ** 2)
        scores[remaining_items] /= candidate_item_squared_norms[remaining_items]
        scores[~remaining_items] = 0
        
        new_item_id = np.argmax(scores)
        assert remaining_items[new_item_id]
        support_items.append(new_item_id)
        support_items_scores.append(scores[new_item_id] / (n_t * n_q))
        remaining_items[new_item_id] = False
        
        new_item = candidate_items[new_item_id].copy()
        new_item -= orthonormed_support_items[:t].T.dot(
            orthonormed_support_items[:t].dot(new_item)
        )
        assert np.allclose((new_item ** 2).sum(), candidate_item_squared_norms[new_item_id], atol=eps)
        new_item /= np.sqrt((new_item ** 2).sum())
        orthonormed_support_items[t] = new_item
        new_coitem = candidate_coitems[new_item_id] / np.sqrt(candidate_item_squared_norms[new_item_id])
        
        coefs = (candidate_items * new_item).sum(axis=1)
        candidate_item_squared_norms -= coefs ** 2
        assert np.all(candidate_item_squared_norms[remaining_items] > -eps)
        
        candidate_coitems -= coefs.reshape((-1, 1)).dot(new_coitem.reshape((1, -1)))
        cocoefs = (candidate_coitems * new_item).sum(axis=1, keepdims=True)
        candidate_coitems -= cocoefs.dot(new_item.reshape((1, -1)))
    return np.array(support_items, dtype=np.int64), np.array(support_items_scores)

In [18]:
ctx.key_games

(array([5557, 6491, 1903,  248, 9446, 4744, 7244, 7104, 1121, 2338, 3864,
        5779, 7137, 2833,  381, 1230, 8071, 1332, 7252, 7008, 3836, 8421,
        2172, 6623, 6689, 8197, 4685, 9364, 5218, 8890, 9380, 1575, 2227,
        3020, 4791, 7807, 1805, 5783, 4066, 9838, 2355, 9200, 4128, 7461,
        5236, 1779, 3046, 5729, 2728, 6605, 9045, 2190, 7592, 1649, 6755,
          89, 6528, 8123, 7599, 2428,  957, 8020, 8998, 5831, 9227,  639,
        4832, 1214,  181, 4184, 4972, 1139, 5304, 6240, 3396, 6563, 3776,
        2262, 4805, 8566, 6095,  102, 9860, 3052, 5068, 3100, 5432, 1326,
        2501,  321,  186, 3516, 5841, 3959, 9276, 8843, 5563, 1502, 5900,
        6003]),
 array([0.34743124, 0.08071299, 0.04348799, 0.04027334, 0.02930198,
        0.0277006 , 0.02327564, 0.01926292, 0.01333025, 0.01233328,
        0.01029388, 0.00868232, 0.00840466, 0.00822039, 0.00820282,
        0.00737349, 0.00651845, 0.00609048, 0.00606077, 0.00505034,
        0.00490937, 0.00482551, 0.00459832, 0.

In [17]:
dataset = DATASETS[0]
    
print ("\n\n\nDATASET = ", dataset)

ctx = EvalContextRelevs(
    load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
    det_attempts=100
)

print("LOADED")

ctx.set_kmeans_games_as_key()




DATASET =  yugioh
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013
LOADED


/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


In [18]:
m = AnnCUR(ctx)
m.fit()
m.get_score("train"), m.get_score("test")

np.mean(results), mse, len(results) =  0.5286150442477875 0.14142281 2260
np.mean(results), mse, len(results) =  0.5080848963474828 0.16278194 1013


(0.5286150442477875, 0.5080848963474828)

In [42]:
t = "test"
key_relevs = m.ctx.get_relevs(t)[:, m.key_cols_idx]
best_relevs = key_relevs.mean(-1)
m_relev = np.median(best_relevs)
best_relevs > m_relev

In [27]:
def get_score(self, t, slice_, topsize = 100, getsize = None):    
    if getsize is None:
        getsize = topsize
    recs = np.array(self.recommend(t))
    trus = self.get_user_scores(t)

    mse = np.mean((recs - trus) ** 2)

    recs = np.argsort(-recs, axis=1)[:, :getsize]
    trus = np.argsort(-trus, axis=1)[:, :topsize]

    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    results = np.array([
        ev(rec, tru) / float(topsize)
        for rec, tru in zip(recs,trus)
    ])
    if slice_ is not None:
        results = results[slice_]

    print("np.mean(results), mse, len(results) = ",
          np.mean(results), mse, len(results))

    return np.mean(results)

In [49]:
get_score(m, "test", best_relevs > m_relev)

np.mean(results), mse, len(results) =  0.6018379446640316 0.16278194 506


0.6018379446640316

In [50]:
get_score(m, "test", best_relevs <= m_relev)

np.mean(results), mse, len(results) =  0.41451676528599607 0.16278194 507


0.41451676528599607

In [16]:
L = 7000
N = 1000
DA = 50

ctx = load(L, raw_path = "stand/log.local.txt", seed=17, det_attempts=DA) # .set_top_games_as_key()
print("LOADED")

ctx.set_kmeans_games_as_key()


# t = ctx.get_relevs("train").T
#t = (t - t.mean()) / t.std()
# ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

N = 100000
m = DssmKnn(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.6095219944834066e-120
2.6095219944834066e-120
101 4769 2088
LOADED


/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


KEY_SET


TypeError: DssmKnn.__init__() missing 1 required positional argument: 'modelid'

In [17]:

print("KEY_SET")

N = 100000
m = DssmKnn(ctx, 6, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

KEY_SET
self.embed_users['train'].shape =  (4769, 100)
self.embed_games.shape =  (16514, 101)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (4769, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.27891089108910894 tf.Tensor(5.577185400552235, shape=(), dtype=float64) 101
slice = key, score = 0.27891089108910894
np.mean(results), mse, len(results) =  0.28855315579786117 tf.Tensor(5.20531790495444, shape=(), dtype=float64) 4769
slice = train, score = 0.28855315579786117
np.mean(results), mse, len(results) =  0.28764367816091957 tf.Tensor(5.110205881602827, shape=(), dtype=float64) 2088
slice = test, score = 0.28764367816091957

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.2894059405940594 tf.Tensor(26.182316398077845, shape=(), dtype=float64) 101
slice = key, score = 0.2894059405940594
np.mean(results), mse, len(results) =  0.309930803103375


KeyboardInterrupt



# AnnCUR vs DualEncoder vs Popular 

In [17]:
L = 7000
N = 1000
DA = 50

In [18]:
ctx = load(L, raw_path = "stand/log.local.logtime2.txt",
           path="log.local.logtime2.true.bin", seed=17, det_attempts=DA)

print("LOADED")

ctx.key_games=ctx.get_kmeans_games(all_from_labels=False)

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.6095219944834066e-120
2.6095219944834066e-120
101 4769 2088
LOADED


/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


### AnnCUR

In [20]:
ma = AnnCUR(ctx)
ma.fit()
ma.get_score("train"), ma.get_score("test")

np.mean(results), mse, len(results) =  0.6313818410568253 0.15723647873628968 4769
np.mean(results), mse, len(results) =  0.6218726053639847 0.1773087967851091 2088


(0.6313818410568253, 0.6218726053639847)

In [22]:
t = "test"
key_relevs = ctx.get_relevs(t)[:, ma.key_cols_idx]
best_relevs = key_relevs.max(-1)
# m_relev = np.median(best_relevs)
# best_relevs > m_relev

array([False, False, False, ..., False, False, False])

In [28]:
slice_ = best_relevs > np.median(best_relevs)
get_score(ma, "test", slice_)

np.mean(results), mse, len(results) =  0.6272318007662835 0.1773087967851091 1044


0.6272318007662835

In [29]:
slice_ = best_relevs <= np.median(best_relevs)
get_score(ma, "test", slice_)

np.mean(results), mse, len(results) =  0.6165134099616858 0.1773087967851091 1044


0.6165134099616858

In [40]:
q = 0
q * step, q * (step + 1)

(0.0, 0.0)

In [44]:
step = 0.25
for q in range(4):
    lo, hi = q * step, (q + 1) * step
    slice_ = (np.quantile(best_relevs, lo) <= best_relevs) & (best_relevs <= np.quantile(best_relevs, hi))
    print("lo, hi = ", lo, hi, np.mean(slice_), np.sum(slice_))
    print(get_score(ma, "test", slice_))

lo, hi =  0.0 0.25 0.25 522
np.mean(results), mse, len(results) =  0.6141187739463603 0.1773087967851091 522
0.6141187739463603
lo, hi =  0.25 0.5 0.25 522
np.mean(results), mse, len(results) =  0.6189080459770114 0.1773087967851091 522
0.6189080459770114
lo, hi =  0.5 0.75 0.25 522
np.mean(results), mse, len(results) =  0.6409195402298852 0.1773087967851091 522
0.6409195402298852
lo, hi =  0.75 1.0 0.25 522
np.mean(results), mse, len(results) =  0.6135440613026819 0.1773087967851091 522
0.6135440613026819


In [43]:
step = 0.1
for q in range(10):
    lo, hi = q * step, (q + 1) * step
    slice_ = (np.quantile(best_relevs, lo) <= best_relevs) & (best_relevs <= np.quantile(best_relevs, hi))
    print("lo, hi = ", lo, hi, np.mean(slice_), np.sum(slice_))
    print(get_score(ma, "test", slice_))

lo, hi =  0.0 0.1 0.10009578544061302 209
np.mean(results), mse, len(results) =  0.590956937799043 0.1773087967851091 209
0.590956937799043
lo, hi =  0.1 0.2 0.10009578544061302 209
np.mean(results), mse, len(results) =  0.6238755980861245 0.1773087967851091 209
0.6238755980861245
lo, hi =  0.2 0.30000000000000004 0.10009578544061302 209
np.mean(results), mse, len(results) =  0.6276555023923446 0.1773087967851091 209
0.6276555023923446
lo, hi =  0.30000000000000004 0.4 0.09961685823754789 208
np.mean(results), mse, len(results) =  0.6250961538461538 0.1773087967851091 208
0.6250961538461538
lo, hi =  0.4 0.5 0.10009578544061302 209
np.mean(results), mse, len(results) =  0.615023923444976 0.1773087967851091 209
0.615023923444976
lo, hi =  0.5 0.6000000000000001 0.10009578544061302 209
np.mean(results), mse, len(results) =  0.6365550239234451 0.1773087967851091 209
0.6365550239234451
lo, hi =  0.6000000000000001 0.7000000000000001 0.09961685823754789 208
np.mean(results), mse, len(result

In [46]:
mp = Popular(ctx)
mp.fit()

In [61]:
mp.get_score("test")

np.mean(results), mse, len(results) =  0.2885536398467433 5.111057961798107 2088


0.2885536398467433

In [48]:
step = 0.5
for q in range(2):
    lo, hi = q * step, (q + 1) * step
    slice_ = (np.quantile(best_relevs, lo) <= best_relevs) & (best_relevs <= np.quantile(best_relevs, hi))
    print("lo, hi = ", lo, hi, np.mean(slice_), np.sum(slice_))
    print(get_score(mp, "test", slice_))

lo, hi =  0.0 0.5 0.5 1044
np.mean(results), mse, len(results) =  0.2879597701149425 5.111057961798107 1044
0.2879597701149425
lo, hi =  0.5 1.0 0.5 1044
np.mean(results), mse, len(results) =  0.28914750957854407 5.111057961798107 1044
0.28914750957854407


In [47]:
step = 0.25
for q in range(4):
    lo, hi = q * step, (q + 1) * step
    slice_ = (np.quantile(best_relevs, lo) <= best_relevs) & (best_relevs <= np.quantile(best_relevs, hi))
    print("lo, hi = ", lo, hi, np.mean(slice_), np.sum(slice_))
    print(get_score(mp, "test", slice_))

lo, hi =  0.0 0.25 0.25 522
np.mean(results), mse, len(results) =  0.2666283524904215 5.111057961798107 522
0.2666283524904215
lo, hi =  0.25 0.5 0.25 522
np.mean(results), mse, len(results) =  0.3092911877394636 5.111057961798107 522
0.3092911877394636
lo, hi =  0.5 0.75 0.25 522
np.mean(results), mse, len(results) =  0.3088505747126437 5.111057961798107 522
0.3088505747126437
lo, hi =  0.75 1.0 0.25 522
np.mean(results), mse, len(results) =  0.2694444444444444 5.111057961798107 522
0.2694444444444444


In [49]:
step = 0.1
for q in range(10):
    lo, hi = q * step, (q + 1) * step
    slice_ = (np.quantile(best_relevs, lo) <= best_relevs) & (best_relevs <= np.quantile(best_relevs, hi))
    print("lo, hi = ", lo, hi, np.mean(slice_), np.sum(slice_))
    print(get_score(mp, "test", slice_))

lo, hi =  0.0 0.1 0.10009578544061302 209
np.mean(results), mse, len(results) =  0.24660287081339716 5.111057961798107 209
0.24660287081339716
lo, hi =  0.1 0.2 0.10009578544061302 209
np.mean(results), mse, len(results) =  0.2731578947368421 5.111057961798107 209
0.2731578947368421
lo, hi =  0.2 0.30000000000000004 0.10009578544061302 209
np.mean(results), mse, len(results) =  0.291244019138756 5.111057961798107 209
0.291244019138756
lo, hi =  0.30000000000000004 0.4 0.09961685823754789 208
np.mean(results), mse, len(results) =  0.30759615384615385 5.111057961798107 208
0.30759615384615385
lo, hi =  0.4 0.5 0.10009578544061302 209
np.mean(results), mse, len(results) =  0.3212918660287082 5.111057961798107 209
0.3212918660287082
lo, hi =  0.5 0.6000000000000001 0.10009578544061302 209
np.mean(results), mse, len(results) =  0.3093301435406699 5.111057961798107 209
0.3093301435406699
lo, hi =  0.6000000000000001 0.7000000000000001 0.09961685823754789 208
np.mean(results), mse, len(result

In [50]:
print("KEY_SET")

N = 10000
m = DssmKnn(ctx, 6, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "learning_rate": 0.001
})

m.fit()
m.get_score("train"), m.get_score("test")

KEY_SET
self.embed_users['train'].shape =  (4769, 100)
self.embed_games.shape =  (16514, 101)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (4769, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.27881188118811884 tf.Tensor(5.617414392735163, shape=(), dtype=float64) 101
slice = key, score = 0.27881188118811884
np.mean(results), mse, len(results) =  0.2889830153071922 tf.Tensor(5.204331371433164, shape=(), dtype=float64) 4769
slice = train, score = 0.2889830153071922
np.mean(results), mse, len(results) =  0.28797413793103455 tf.Tensor(5.1044563545092405, shape=(), dtype=float64) 2088
slice = test, score = 0.28797413793103455

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.3280198019801981 tf.Tensor(18.903603870025027, shape=(), dtype=float64) 101
slice = key, score = 0.3280198019801981
np.mean(results), mse, len(results) =  0.348957852799328

(0.453233382260432, 0.45029214559386976)

In [52]:
step = 0.5
for q in range(2):
    lo, hi = q * step, (q + 1) * step
    slice_ = (np.quantile(best_relevs, lo) <= best_relevs) & (best_relevs <= np.quantile(best_relevs, hi))
    print("lo, hi = ", lo, hi, np.mean(slice_), np.sum(slice_))
    print(get_score(m, "test", slice_))

lo, hi =  0.0 0.5 0.5 1044
np.mean(results), mse, len(results) =  0.435507662835249 975.1151876759925 1044
0.435507662835249
lo, hi =  0.5 1.0 0.5 1044
np.mean(results), mse, len(results) =  0.46507662835249036 975.1151876759925 1044
0.46507662835249036


In [53]:
step = 0.25
for q in range(4):
    lo, hi = q * step, (q + 1) * step
    slice_ = (np.quantile(best_relevs, lo) <= best_relevs) & (best_relevs <= np.quantile(best_relevs, hi))
    print("lo, hi = ", lo, hi, np.mean(slice_), np.sum(slice_))
    print(get_score(m, "test", slice_))

lo, hi =  0.0 0.25 0.25 522
np.mean(results), mse, len(results) =  0.4370689655172414 975.1151876759925 522
0.4370689655172414
lo, hi =  0.25 0.5 0.25 522
np.mean(results), mse, len(results) =  0.43394636015325666 975.1151876759925 522
0.43394636015325666
lo, hi =  0.5 0.75 0.25 522
np.mean(results), mse, len(results) =  0.4802298850574713 975.1151876759925 522
0.4802298850574713
lo, hi =  0.75 1.0 0.25 522
np.mean(results), mse, len(results) =  0.4499233716475096 975.1151876759925 522
0.4499233716475096


In [54]:
step = 0.1
for q in range(10):
    lo, hi = q * step, (q + 1) * step
    slice_ = (np.quantile(best_relevs, lo) <= best_relevs) & (best_relevs <= np.quantile(best_relevs, hi))
    print("lo, hi = ", lo, hi, np.mean(slice_), np.sum(slice_))
    print(get_score(m, "test", slice_))

lo, hi =  0.0 0.1 0.10009578544061302 209
np.mean(results), mse, len(results) =  0.39511961722488037 975.1151876759925 209
0.39511961722488037
lo, hi =  0.1 0.2 0.10009578544061302 209
np.mean(results), mse, len(results) =  0.4669377990430622 975.1151876759925 209
0.4669377990430622
lo, hi =  0.2 0.30000000000000004 0.10009578544061302 209
np.mean(results), mse, len(results) =  0.4380861244019139 975.1151876759925 209
0.4380861244019139
lo, hi =  0.30000000000000004 0.4 0.09961685823754789 208
np.mean(results), mse, len(results) =  0.430625 975.1151876759925 208
0.430625
lo, hi =  0.4 0.5 0.10009578544061302 209
np.mean(results), mse, len(results) =  0.4467464114832536 975.1151876759925 209
0.4467464114832536
lo, hi =  0.5 0.6000000000000001 0.10009578544061302 209
np.mean(results), mse, len(results) =  0.47674641148325353 975.1151876759925 209
0.47674641148325353
lo, hi =  0.6000000000000001 0.7000000000000001 0.09961685823754789 208
np.mean(results), mse, len(results) =  0.4901442307

In [57]:
print("KEY_SET")

N = 10000
mc = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "learning_rate": 0.001
})

mc.fit()
mc.get_score("train"), mc.get_score("test")

KEY_SET
self.embed_users['train'].shape =  (4769, 100)
self.embed_games.shape =  (16514, 101)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (4769, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0904950495049505 tf.Tensor(53.38301092359495, shape=(), dtype=float64) 101
slice = key, score = 0.0904950495049505
np.mean(results), mse, len(results) =  0.08854267141958481 tf.Tensor(53.76124840873226, shape=(), dtype=float64) 4769
slice = train, score = 0.08854267141958481
np.mean(results), mse, len(results) =  0.0884051724137931 tf.Tensor(52.903494768789734, shape=(), dtype=float64) 2088
slice = test, score = 0.0884051724137931

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.4487128712871288 tf.Tensor(666.2956263554017, shape=(), dtype=float64) 101
slice = key, score = 0.4487128712871288
np.mean(results), mse, len(results) =  0.46273222897882155 

(0.5536779198993499, 0.5481561302681993)

In [58]:
step = 0.5
for q in range(2):
    lo, hi = q * step, (q + 1) * step
    slice_ = (np.quantile(best_relevs, lo) <= best_relevs) & (best_relevs <= np.quantile(best_relevs, hi))
    print("lo, hi = ", lo, hi, np.mean(slice_), np.sum(slice_))
    print(get_score(mc, "test", slice_))

lo, hi =  0.0 0.5 0.5 1044
np.mean(results), mse, len(results) =  0.5287835249042145 9990.20667689902 1044
0.5287835249042145
lo, hi =  0.5 1.0 0.5 1044
np.mean(results), mse, len(results) =  0.5675287356321839 9990.20667689902 1044
0.5675287356321839


In [59]:
step = 0.25
for q in range(4):
    lo, hi = q * step, (q + 1) * step
    slice_ = (np.quantile(best_relevs, lo) <= best_relevs) & (best_relevs <= np.quantile(best_relevs, hi))
    print("lo, hi = ", lo, hi, np.mean(slice_), np.sum(slice_))
    print(get_score(mc, "test", slice_))

lo, hi =  0.0 0.25 0.25 522
np.mean(results), mse, len(results) =  0.5045977011494253 9990.20667689902 522
0.5045977011494253
lo, hi =  0.25 0.5 0.25 522
np.mean(results), mse, len(results) =  0.5529693486590038 9990.20667689902 522
0.5529693486590038
lo, hi =  0.5 0.75 0.25 522
np.mean(results), mse, len(results) =  0.5807471264367816 9990.20667689902 522
0.5807471264367816
lo, hi =  0.75 1.0 0.25 522
np.mean(results), mse, len(results) =  0.5543103448275862 9990.20667689902 522
0.5543103448275862


In [60]:
step = 0.1
for q in range(10):
    lo, hi = q * step, (q + 1) * step
    slice_ = (np.quantile(best_relevs, lo) <= best_relevs) & (best_relevs <= np.quantile(best_relevs, hi))
    print("lo, hi = ", lo, hi, np.mean(slice_), np.sum(slice_))
    print(get_score(mc, "test", slice_))

lo, hi =  0.0 0.1 0.10009578544061302 209
np.mean(results), mse, len(results) =  0.44961722488038275 9990.20667689902 209
0.44961722488038275
lo, hi =  0.1 0.2 0.10009578544061302 209
np.mean(results), mse, len(results) =  0.5245933014354067 9990.20667689902 209
0.5245933014354067
lo, hi =  0.2 0.30000000000000004 0.10009578544061302 209
np.mean(results), mse, len(results) =  0.5595693779904306 9990.20667689902 209
0.5595693779904306
lo, hi =  0.30000000000000004 0.4 0.09961685823754789 208
np.mean(results), mse, len(results) =  0.5561538461538462 9990.20667689902 208
0.5561538461538462
lo, hi =  0.4 0.5 0.10009578544061302 209
np.mean(results), mse, len(results) =  0.5541148325358852 9990.20667689902 209
0.5541148325358852
lo, hi =  0.5 0.6000000000000001 0.10009578544061302 209
np.mean(results), mse, len(results) =  0.5779904306220096 9990.20667689902 209
0.5779904306220096
lo, hi =  0.6000000000000001 0.7000000000000001 0.09961685823754789 208
np.mean(results), mse, len(results) =  

https://docs.google.com/spreadsheets/d/19MvMIZvuybW4qKtahANZQswBzBC980knQyY6XIf42XI/edit#gid=0

# downsampled keys

16514

In [65]:
r = ctx.get_relevs("train").T

In [82]:
r[idxs == 1].shape

(4129, 4769)

In [103]:
for dataset in DATASETS:
    DSK = 4
    print ("\n\n\nDATASET = ", dataset)

    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100
    )

    print("LOADED")

    #ctx.set_kmeans_games_as_key()

    X = ctx.get_relevs("train").T
    mask = [0] * DSK
    mask[0] = 1

    idxs = np.array((mask * X.shape[0])[:X.shape[0]])
    X = X[idxs == 1]
    all_from_labels = True
    
    k_func = (
        (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
        if not all_from_labels else
        (lambda C : from_labels(X, C.labels_))
    )
    K_KMeans = k_func(
        KMeans(n_clusters=ctx.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
    )
    K_KMeans = [x * DSK for x in K_KMeans]

    ctx.key_games = np.array(sorted(K_KMeans))
    
    m_ = AnnCUR(ctx)
    m_.fit()
    m_.get_score("train"), m_.get_score("test")




DATASET =  yugioh
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013
LOADED


/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


np.mean(results), mse, len(results) =  0.5303230088495575 0.14509949 2260
np.mean(results), mse, len(results) =  0.511243830207305 0.17232959 1013



DATASET =  pro_wrestling
Loading file pro_wrestling/ment_to_ent_scores_n_m_392_n_e_10133_all_layers_False1000.pkl
Loading file pro_wrestling/ment_to_ent_scores_n_m_500_n_e_10133_all_layers_False0.pkl
Loading file pro_wrestling/ment_to_ent_scores_n_m_500_n_e_10133_all_layers_False500.pkl
Loaded shape =  (1392, 10133)
Shuffling... (seed = 42)
updated det (1, 5.124810435461604e-33 -> 1.0217253487036523e-31)
updated det (2, 1.0217253487036523e-31 -> 1.2030961336739456e-31)
updated det (11, 1.2030961336739456e-31 -> 7.606045638566237e-29)
updated det (60, 7.606045638566237e-29 -> 3.735096148850118e-27)
Best det =  3.735096e-27
Current de =  3.735096e-27
100 873 418
LOADED


/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


np.mean(results), mse, len(results) =  0.5283619702176403 0.047802683 873
np.mean(results), mse, len(results) =  0.4684928229665072 0.06632713 418



DATASET =  star_trek
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False3000.pkl
Loading file star_trek/ment_to_ent_scores_n_m_27_n_e_34430_all_layers_False4200.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False0.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False3200.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False2200.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False200.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False1800.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False2600.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False800.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers

...

np.mean(results), mse, len(results) =  0.3351347567378369 0.032007918 2857
np.mean(results), mse, len(results) =  0.3101260835303389 0.037507575 1269



DATASET =  doctor_who
Loading file doctor_who/ment_to_ent_scores_n_m_200_n_e_40281_all_layers_False1800.pkl
Loading file doctor_who/ment_to_ent_scores_n_m_200_n_e_40281_all_layers_False0400.pkl
Loading file doctor_who/ment_to_ent_scores_n_m_200_n_e_40281_all_layers_False3800.pkl
Loading file doctor_who/ment_to_ent_scores_n_m_200_n_e_40281_all_layers_False2600.pkl
Loading file doctor_who/ment_to_ent_scores_n_m_200_n_e_40281_all_layers_False0000.pkl
Loading file doctor_who/ment_to_ent_scores_n_m_200_n_e_40281_all_layers_False2000.pkl
Loading file doctor_who/ment_to_ent_scores_n_m_200_n_e_40281_all_layers_False2200.pkl
Loading file doctor_who/ment_to_ent_scores_n_m_200_n_e_40281_all_layers_False1400.pkl
Loading file doctor_who/ment_to_ent_scores_n_m_200_n_e_40281_all_layers_False2400.pkl
Loading file doctor_who/ment_to_ent_scores_n_m_200_

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


np.mean(results), mse, len(results) =  0.27357169321971103 0.030687623 2699
np.mean(results), mse, len(results) =  0.2513666666666667 0.040551078 1200



DATASET =  military
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False2200.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0500.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0400.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1800.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False2000.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1600.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0600.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1200.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0900.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_a

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


np.mean(results), mse, len(results) =  0.34594680177327425 0.0073913024 1579
np.mean(results), mse, len(results) =  0.2853888888888889 0.009821491 720


In [102]:
sorted(K_KMeans)

[20,
 176,
 224,
 356,
 484,
 660,
 784,
 1040,
 1060,
 1124,
 1412,
 1432,
 1444,
 1548,
 1564,
 1580,
 1604,
 1680,
 1844,
 1912,
 1928,
 1972,
 2176,
 2392,
 2392,
 2412,
 2720,
 2808,
 2828,
 2952,
 2976,
 3152,
 3444,
 3444,
 3460,
 3556,
 3604,
 3652,
 3700,
 3768,
 3896,
 3972,
 3992,
 4020,
 4152,
 4320,
 4392,
 5072,
 5096,
 5196,
 5328,
 5352,
 5456,
 5516,
 5528,
 5756,
 5760,
 5776,
 5900,
 5944,
 6112,
 6172,
 6356,
 6608,
 6724,
 6804,
 7008,
 7092,
 7108,
 7296,
 7308,
 7332,
 7492,
 7876,
 7908,
 7908,
 8012,
 8032,
 8404,
 8464,
 8532,
 8536,
 8544,
 8564,
 8828,
 8984,
 9024,
 9100,
 9104,
 9128,
 9188,
 9288,
 9376,
 9500,
 9632,
 9700,
 9756,
 9780,
 9872,
 9944]